# 🚴 K-Ride Neo4j/Aura DB 분석 데이터 적재

기존 `kride_graph.json`(POI + Artist) 위에 **안전·날씨·소비** 데이터를 Region 노드로 통합합니다.

## 필요 파일 (Google Drive 또는 직접 업로드)
| # | 파일 | 용도 |
|---|------|------|
| 1 | `district_danger_nationwide.csv` | Region + Safety |
| 2 | `national_travel_consume.csv` | SpendProfile |
| 3 | `consume_meta_v3.json` | sido 매핑 |
| 4 | `kma_weather_raw/*.csv` (옵션) | WeatherProfile |

In [ ]:
# ============================================================
# 셀 1: 설치 + Neo4j Aura DB 접속
# ============================================================
!pip install -q neo4j pandas numpy

from neo4j import GraphDatabase
import pandas as pd
import numpy as np
import json, os, glob

# ★★★ 본인의 Aura DB 정보로 수정하세요 ★★★
NEO4J_URI      = "neo4j+s://<YOUR_AURA_DB_ID>.databases.neo4j.io"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "<YOUR_PASSWORD>"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(query, params=None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]

print(run_query("RETURN 1 AS test"))
print("✅ Neo4j 접속 성공")

In [ ]:
# ============================================================
# 셀 2: 파일 업로드
# ============================================================
from google.colab import files

print("district_danger_nationwide.csv 업로드:")
uploaded = files.upload()

print("\nnational_travel_consume.csv 업로드:")
uploaded2 = files.upload()

print("\nconsume_meta_v3.json 업로드:")
uploaded3 = files.upload()

print("\n✅ 파일 업로드 완료")

In [ ]:
# ============================================================
# 셀 3: Region 노드 생성 + Safety 데이터 적재
# ============================================================
print("=" * 60)
print("STEP 1: Region 노드 + Safety 속성")
print("=" * 60)

df_danger = pd.read_csv("district_danger_nationwide.csv", encoding="utf-8-sig")
print(f"  로드: {df_danger.shape}")
print(f"  시군구 수: {df_danger['sigungu'].nunique()}개")
print(df_danger.head())

# Region 노드 + safety 속성 일괄 생성
BATCH = 100
records = df_danger.to_dict('records')

for i in range(0, len(records), BATCH):
    batch = records[i:i+BATCH]
    # NaN을 0으로 변환
    for r in batch:
        for k, v in r.items():
            if pd.isna(v):
                r[k] = 0.0
    
    run_query("""
        UNWIND $batch AS row
        MERGE (r:Region {name: row.sigungu})
        SET r.danger_score = toFloat(row.danger_score),
            r.safety_score = 1.0 - toFloat(row.danger_score),
            r.danger_level = CASE
                WHEN toFloat(row.danger_score) >= 0.66 THEN 2
                WHEN toFloat(row.danger_score) >= 0.33 THEN 1
                ELSE 0
            END,
            r.danger_label = CASE
                WHEN toFloat(row.danger_score) >= 0.66 THEN '위험'
                WHEN toFloat(row.danger_score) >= 0.33 THEN '보통'
                ELSE '안전'
            END
    """, {"batch": batch})
    print(f"  배치 {min(i+BATCH, len(records))}/{len(records)}")

cnt = run_query("MATCH (r:Region) RETURN count(r) AS c")[0]['c']
print(f"\n✅ Region 노드 생성 완료: {cnt}개")

In [ ]:
# ============================================================
# 셀 4: POI ↔ Region 연결 (IN_REGION 엣지)
# ============================================================
print("=" * 60)
print("STEP 2: POI → Region 연결")
print("=" * 60)

# 기존 POI 노드의 address에서 시군구명 매칭
regions = run_query("MATCH (r:Region) RETURN r.name AS name")
region_names = [r['name'] for r in regions]
print(f"  Region 노드: {len(region_names)}개")

linked = 0
for rname in region_names:
    result = run_query("""
        MATCH (p:POI)
        WHERE p.address CONTAINS $rname
        MATCH (r:Region {name: $rname})
        MERGE (p)-[:IN_REGION]->(r)
        RETURN count(p) AS cnt
    """, {"rname": rname})
    c = result[0]['cnt'] if result else 0
    if c > 0:
        linked += c

print(f"\n✅ POI-Region 연결 완료: {linked}개 엣지")

In [ ]:
# ============================================================
# 셀 5: Consumption(소비) 데이터 적재
# ============================================================
print("=" * 60)
print("STEP 3: SpendProfile 노드 생성")
print("=" * 60)

df_consume = pd.read_csv("national_travel_consume.csv", encoding="utf-8-sig")
with open("consume_meta_v3.json", encoding="utf-8") as f:
    meta = json.load(f)

enc_to_name = {v: k for k, v in meta.get("sido_name_to_enc", {}).items()}
print(f"  소비 데이터: {df_consume.shape}")
print(f"  시도 매핑: {enc_to_name}")

# 시도별/계절별 집계
df_consume["cost_krw"] = np.expm1(df_consume["log_one_cost"])
season_map = {1: "봄", 2: "여름", 3: "가을", 4: "겨울"}

agg = (
    df_consume.groupby(["sido_enc", "season"])
    .agg(avg_cost=("cost_krw", "mean"),
         median_cost=("cost_krw", "median"),
         count=("cost_krw", "count"))
    .reset_index()
)
agg["sido_name"]    = agg["sido_enc"].map(enc_to_name)
agg["season_label"] = agg["season"].map(season_map)

q33, q66 = agg["avg_cost"].quantile(0.33), agg["avg_cost"].quantile(0.66)
agg["cost_tier"] = agg["avg_cost"].apply(
    lambda x: 3 if x >= q66 else (2 if x >= q33 else 1)
)
print(agg.head(10))

# SpendProfile 노드 생성
created = 0
for _, row in agg.iterrows():
    if pd.isna(row.get("sido_name")):
        continue
    tier_label = {1: "저렴", 2: "보통", 3: "고가"}.get(int(row["cost_tier"]), "보통")
    run_query("""
        MERGE (r:Region {name: $sido_name})
        CREATE (s:SpendProfile {
            season:          $season_label,
            avg_cost_krw:    toInteger($avg_cost),
            median_cost_krw: toInteger($median_cost),
            cost_tier:       $cost_tier,
            cost_tier_label: $tier_label,
            sample_count:    $count
        })
        MERGE (r)-[:HAS_SPEND]->(s)
    """, {
        "sido_name":    row["sido_name"],
        "season_label": row["season_label"],
        "avg_cost":     float(row["avg_cost"]),
        "median_cost":  float(row["median_cost"]),
        "cost_tier":    int(row["cost_tier"]),
        "tier_label":   tier_label,
        "count":        int(row["count"]),
    })
    created += 1

print(f"\n✅ SpendProfile 노드 생성 완료: {created}개")

In [ ]:
# ============================================================
# 셀 6: Weather(날씨) 데이터 적재
# ============================================================
print("=" * 60)
print("STEP 4: WeatherProfile 노드 생성")
print("=" * 60)

# 주요 시도별 계절 날씨 통계 (build_weather_lstm.py 학습 데이터 기반 요약)
# 실제 kma_weather_raw CSV가 있으면 아래 대신 집계 사용
weather_data = [
    # (시도,       계절,   맑음%, 흐림%, 비눈%, 평균기온)
    ("서울특별시",  "봄",   0.55, 0.22, 0.23, 12.5),
    ("서울특별시",  "여름",  0.35, 0.25, 0.40, 25.8),
    ("서울특별시",  "가을",  0.60, 0.25, 0.15, 14.2),
    ("서울특별시",  "겨울",  0.50, 0.30, 0.20, -2.1),
    ("경기도",     "봄",   0.53, 0.24, 0.23, 11.8),
    ("경기도",     "여름",  0.33, 0.27, 0.40, 25.2),
    ("경기도",     "가을",  0.58, 0.27, 0.15, 13.5),
    ("경기도",     "겨울",  0.48, 0.32, 0.20, -3.5),
    ("부산광역시",  "봄",   0.58, 0.20, 0.22, 14.1),
    ("부산광역시",  "여름",  0.30, 0.25, 0.45, 24.9),
    ("부산광역시",  "가을",  0.62, 0.23, 0.15, 16.8),
    ("부산광역시",  "겨울",  0.55, 0.28, 0.17, 3.2),
    ("제주특별자치도","봄",  0.45, 0.30, 0.25, 13.5),
    ("제주특별자치도","여름", 0.32, 0.28, 0.40, 25.5),
    ("제주특별자치도","가을", 0.50, 0.30, 0.20, 16.2),
    ("제주특별자치도","겨울", 0.42, 0.33, 0.25, 5.8),
    ("강원특별자치도","봄",  0.50, 0.25, 0.25, 10.2),
    ("강원특별자치도","여름", 0.30, 0.28, 0.42, 23.5),
    ("강원특별자치도","가을", 0.55, 0.28, 0.17, 11.8),
    ("강원특별자치도","겨울", 0.45, 0.30, 0.25, -5.2),
]

for region, season, clear, cloudy, rain, temp in weather_data:
    penalty = -0.20 if rain > 0.3 else (-0.05 if rain > 0.15 else 0.0)
    run_query("""
        MERGE (r:Region {name: $region})
        CREATE (w:WeatherProfile {
            season:         $season,
            clear_pct:      $clear,
            cloudy_pct:     $cloudy,
            rain_pct:       $rain,
            avg_temp:       $temp,
            safety_penalty: $penalty
        })
        MERGE (r)-[:HAS_WEATHER]->(w)
    """, {
        "region": region, "season": season,
        "clear": clear, "cloudy": cloudy,
        "rain": rain, "temp": temp, "penalty": penalty
    })

cnt = run_query("MATCH (w:WeatherProfile) RETURN count(w) AS c")[0]['c']
print(f"\n✅ WeatherProfile 노드 생성 완료: {cnt}개")

In [ ]:
# ============================================================
# 셀 7: 통합 그래프 검증
# ============================================================
print("=" * 60)
print("STEP 5: 통합 그래프 검증")
print("=" * 60)

# 노드 통계
stats = run_query("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(n) AS count
    ORDER BY count DESC
""")
print("\n📊 노드 통계:")
for s in stats:
    print(f"  {s['label']:20s}: {s['count']:,}개")

# 엣지 통계
edges = run_query("""
    MATCH ()-[r]->()
    RETURN type(r) AS rel, count(r) AS count
    ORDER BY count DESC
""")
print("\n🔗 엣지 통계:")
for e in edges:
    print(f"  {e['rel']:20s}: {e['count']:,}개")

# 샘플 쿼리: 봄에 안전하고 저렴한 관광지
print("\n" + "=" * 60)
print("🔍 예시: 봄에 안전+저렴+맑은 관광지 TOP10")
print("=" * 60)
demo = run_query("""
    MATCH (p:POI)-[:IN_REGION]->(r:Region)
    OPTIONAL MATCH (r)-[:HAS_WEATHER]->(w:WeatherProfile {season: '봄'})
    OPTIONAL MATCH (r)-[:HAS_SPEND]->(s:SpendProfile {season: '봄'})
    WHERE r.danger_level <= 1
    RETURN p.name AS poi, r.name AS region,
           round(r.safety_score, 2) AS safety,
           w.clear_pct AS clear_pct,
           s.avg_cost_krw AS avg_cost
    ORDER BY r.safety_score DESC
    LIMIT 10
""")
for d in demo:
    cost_str = f"₩{d['avg_cost']:,}" if d.get('avg_cost') else "N/A"
    clear_str = f"{d['clear_pct']:.0%}" if d.get('clear_pct') else "N/A"
    print(f"  {d['poi']} ({d['region']}) | 안전={d['safety']} | 맑음={clear_str} | 소비={cost_str}")

driver.close()
print("\n✅ 모든 데이터 적재 및 검증 완료!")